# Generating reference points for a group of decision makers

When each decision maker (DM) in a group gives an aspiration point, `desdeo.tools.group_reference_points` generates
reference points inside the convex hull of those points. Passing them to the Iterative Pareto Representer (IPR, see
[How to generate a representative set of solutions](../IPR/)) gives Pareto optimal solutions that lie between the DMs'
aspirations. This notebook shows one such round on a problem with three objectives.

In [ ]:
import numpy as np
import plotly.graph_objects as go
import polars as pl
from IPython.display import clear_output

from desdeo.problem.testproblems import forest_problem
from desdeo.tools import GurobipySolver, payoff_table_method
from desdeo.tools.group_reference_points import (
    denormalize_reference_point,
    generate_group_reference_points,
    normalize_objective_vectors,
)
from desdeo.tools.iterative_pareto_representer import _EvaluatedPoint, choose_reference_point
from desdeo.tools.scalarization import add_asf_diff

The problem is the forest management problem from the IPR guide. Its three objectives are all maximized: net present
value (`f_1`), wood stock volume (`f_2`) and harvest value (`f_3`). The ideal and nadir points are needed to normalize
the aspirations, and here they come from the payoff table.

In [ ]:
problem = forest_problem(
    simulation_results="../../tests/data/alternatives_290124.csv",
    treatment_key="../../tests/data/alternatives_key_290124.csv",
    holding=5,
    comparing=True,
)
ideal, nadir = payoff_table_method(problem=problem)
clear_output()
problem = problem.update_ideal_and_nadir(new_ideal=ideal, new_nadir=nadir)
symbols = [objective.symbol for objective in problem.objectives]

pl.DataFrame([ideal, nadir]).insert_column(0, pl.Series("point", ["ideal", "nadir"]))

Each DM gives an aspiration point in the original units. Several DMs may give the same point, and some points may
lie inside the hull of the others. Here DM 5 repeats DM 1.

In [ ]:
aspirations = [
    {"f_1": 70000.0, "f_2": -1500.0, "f_3": 130000.0},
    {"f_1": 35000.0, "f_2": 800.0, "f_3": 30000.0},
    {"f_1": 60000.0, "f_2": 500.0, "f_3": 60000.0},
    {"f_1": 55000.0, "f_2": 0.0, "f_3": 75000.0},
    {"f_1": 70000.0, "f_2": -1500.0, "f_3": 130000.0},
]
dm_names = [f"DM {i + 1}" for i in range(len(aspirations))]

pl.DataFrame(aspirations).insert_column(0, pl.Series("DM", dm_names))

`generate_group_reference_points` returns the candidate reference points for IPR. They are in IPR's normalized
space (ideal at 0, nadir at 1, every objective minimized) and each row sums to the number of objectives.
`denormalize_reference_point` converts a row back to original units. The candidates lie on a plane through the
nadir, which is where IPR expects them, so in original units a candidate can be worse than the nadir in some
objectives, as in the example below.

In [ ]:
reference_points = generate_group_reference_points(problem, aspirations, num_points=10_000, seed=0)
print(f"Shape: {reference_points.shape}")
denormalize_reference_point(problem, reference_points[0])

The IPR loop is the one from the IPR guide, with the group's reference points as the candidates. IPR discards
candidates that would lead back to a solution it has already found, and `choose_reference_point` raises an error once
none are left. A hull this size supports the 30 iterations used here.

In [ ]:
# choose_reference_point picks the first candidate with NumPy's global generator.
np.random.seed(0)  # noqa: NPY002
evaluated_points: list[_EvaluatedPoint] = []
for _ in range(30):
    reference_point, _ = choose_reference_point(reference_points, evaluated_points or None)
    scalarized, target = add_asf_diff(problem, "asf", denormalize_reference_point(problem, reference_point))
    objectives = GurobipySolver(scalarized, options={"OutputFlag": 0}).solve(target).optimal_objectives
    targets = normalize_objective_vectors(problem, [objectives])[0]
    evaluated_points.append(
        _EvaluatedPoint(
            reference_point=dict(zip(symbols, reference_point.tolist(), strict=True)),
            targets=dict(zip(symbols, targets.tolist(), strict=True)),
            objectives=objectives,
        )
    )

The figure shows the DMs' aspirations, the reference points IPR used (in original units), and the solutions found
from them. Hovering shows the values.

In [ ]:
used = np.array(
    [
        list(denormalize_reference_point(problem, np.array(list(p.reference_point.values()))).values())
        for p in evaluated_points
    ]
)
solutions = pl.DataFrame([point.objectives for point in evaluated_points])

# Identical aspirations share one marker and one label.
labels: dict[tuple, list[str]] = {}
for name, aspiration in zip(dm_names, aspirations, strict=True):
    labels.setdefault(tuple(aspiration.values()), []).append(name)
aspiration_points = np.array(list(labels))

fig = go.Figure()
fig.add_trace(
    go.Scatter3d(
        x=used[:, 0],
        y=used[:, 1],
        z=used[:, 2],
        mode="markers",
        marker={"size": 3, "color": "#898781"},
        name="Reference points used",
    )
)
fig.add_trace(
    go.Scatter3d(
        x=solutions["f_1"],
        y=solutions["f_2"],
        z=solutions["f_3"],
        mode="markers",
        marker={"size": 5, "color": "#eb6834"},
        name="Solutions",
    )
)
fig.add_trace(
    go.Scatter3d(
        x=aspiration_points[:, 0],
        y=aspiration_points[:, 1],
        z=aspiration_points[:, 2],
        mode="markers+text",
        text=[", ".join(names) for names in labels.values()],
        marker={"size": 6, "color": "#2a78d6"},
        name="DM aspirations",
    )
)
fig.update_layout(
    template="plotly_white",
    height=650,
    margin={"l": 0, "r": 0, "t": 10, "b": 0},
    legend={"orientation": "h"},
    scene={f"{axis}axis_title": objective.name for axis, objective in zip("xyz", problem.objectives, strict=True)},
)
fig.layout.scene.camera.projection.type = "orthographic"
fig.show(renderer="notebook")

The solutions as a table:

In [ ]:
solutions